# 04 — CNN Model

Train a CNN on spectrogram images.

1. Show sample augmented images from `FoxSpectrogramDataset`.
2. Run `train_cnn()` and display training curves.
3. Load best checkpoint and display per-class metrics.

> If real spectrograms are unavailable, synthetic images are generated.

In [ ]:
import os, sys, shutil, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.dataset import FoxSpectrogramDataset
from src.cnn_model import FoxCNN
from src.train_cnn import train_cnn

SR = 22050
MANIFEST = os.path.join(PROJECT_ROOT, "data", "manifest.csv")
SPEC_DIR = os.path.join(PROJECT_ROOT, "data", "spectrograms")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models", "cnn")

print(f"Project root: {PROJECT_ROOT}")

## Helper: ensure spectrograms exist (or create synthetic ones)

In [ ]:
USE_REAL_DATA = os.path.isfile(MANIFEST) and os.path.isdir(SPEC_DIR)
TEMP_DIR = None

if USE_REAL_DATA:
    manifest_csv = MANIFEST
    spec_dir = SPEC_DIR
    print("Using real spectrogram data.")
else:
    print("⚠️  Real data not found — generating synthetic spectrograms.")
    TEMP_DIR = tempfile.mkdtemp(prefix="cnn_nb_")
    manifest_csv = os.path.join(TEMP_DIR, "manifest.csv")
    spec_dir = os.path.join(TEMP_DIR, "specs")

    rows = []
    rng = np.random.RandomState(42)
    for label in ["fox", "nonfox"]:
        label_dir = os.path.join(spec_dir, label)
        os.makedirs(label_dir, exist_ok=True)
        for j in range(20):
            fid = f"{label}_{j:03d}"
            arr = (rng.rand(128, 128) * 255).astype(np.uint8)
            if label == "fox":
                arr[:64, :] = np.clip(arr[:64, :].astype(int) + 30, 0, 255).astype(np.uint8)
            img = Image.fromarray(arr, mode="L").convert("RGB")
            img.save(os.path.join(label_dir, f"{fid}.png"))
            rows.append({"file_id": fid, "clip_path": f"clips/{fid}.wav", "label": label})
    pd.DataFrame(rows).to_csv(manifest_csv, index=False)
    print(f"  Created {len(rows)} synthetic PNGs in {spec_dir}")

## 1. Sample augmented images from `FoxSpectrogramDataset`

In [ ]:
try:
    train_ds = FoxSpectrogramDataset(
        manifest_csv, spec_dir, split="train",
        img_size=(128, 128), augment=True,
    )
    print(f"Train dataset: {len(train_ds)} samples")

    n_show = min(8, len(train_ds))
    cols = min(4, n_show)
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.array(axes).flatten()

    label_names = {0: "nonfox", 1: "fox"}
    for i in range(n_show):
        img_t, lab = train_ds[i]
        img_np = img_t.permute(1, 2, 0).numpy()
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np, 0, 1)
        axes[i].imshow(img_np)
        axes[i].set_title(label_names[lab], fontsize=9)
        axes[i].axis("off")
    for j in range(n_show, len(axes)):
        axes[j].axis("off")

    fig.suptitle("Augmented training samples", fontsize=13)
    fig.tight_layout()
    plt.show()
except Exception as e:
    print(f"⚠️  Could not display augmented images: {e}")

## 2. Train the CNN and display training curves

In [ ]:
cnn_model_dir = os.path.join(TEMP_DIR or MODEL_DIR, "cnn_run")

try:
    best_pt = train_cnn(
        manifest_csv=manifest_csv,
        spec_dir=spec_dir,
        model_dir=cnn_model_dir,
        backbone="efficientnet_b0",
        pretrained=not bool(TEMP_DIR),
        epochs=5 if TEMP_DIR else 15,
        batch_size=8,
        lr=1e-3,
        patience=3,
        img_size=(128, 128),
    )
    print(f"\n✓ Best checkpoint: {best_pt}")
except Exception as e:
    print(f"⚠️  Training failed: {e}")
    best_pt = None

In [ ]:
# Display training curves saved by train_cnn
curves_path = os.path.join(cnn_model_dir, "training_curves.png")
if os.path.isfile(curves_path):
    img = Image.open(curves_path)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Training Curves")
    plt.show()
else:
    print("⚠️  training_curves.png not found.")

## 3. Load best checkpoint & per-class metrics

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader

try:
    if best_pt is None:
        raise FileNotFoundError("No checkpoint available")

    device = "cuda" if torch.cuda.is_available() else (
        "mps" if torch.backends.mps.is_available() else "cpu")
    dev = torch.device(device)

    ckpt = torch.load(best_pt, map_location=dev, weights_only=False)
    model = FoxCNN(backbone=ckpt.get("backbone", "efficientnet_b0"),
                   pretrained=False, num_classes=2)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(dev).eval()

    test_ds = FoxSpectrogramDataset(
        manifest_csv, spec_dir, split="test",
        img_size=(128, 128), augment=False)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

    all_labels, all_preds = [], []
    with torch.no_grad():
        for imgs, labs in test_loader:
            logits = model(imgs.to(dev))
            preds = logits.argmax(dim=1).cpu().tolist()
            all_labels.extend(labs.tolist())
            all_preds.extend(preds)

    print(classification_report(
        all_labels, all_preds,
        labels=[0, 1], target_names=["nonfox", "fox"], zero_division=0))
except Exception as e:
    print(f"⚠️  Checkpoint evaluation skipped: {e}")

In [ ]:
# Clean up temporary synthetic data
if TEMP_DIR and os.path.isdir(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)
    print(f"Cleaned up {TEMP_DIR}")